<a href="https://colab.research.google.com/github/RuchiKalkandha/DeepSeek_architecture/blob/main/MOE_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(42)

Defining each expert as a neural network

In [42]:
# Expert Module
class Expert(nn.Module) :
  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd, n_embd),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)

Implementing the router

In [43]:
num_experts = 3
top_k = 2
n_embd = 8

mh_output = torch.randn(1,4,n_embd)
topkgate_linear = nn.Linear(n_embd, num_experts)
logits = topkgate_linear(mh_output)
print(logits)
#  logits matrix = expert selector matrix

tensor([[[ 0.0238, -0.2771, -0.5070],
         [-0.5727, -0.9081,  0.1839],
         [ 0.8137,  0.1781,  1.5661],
         [ 0.6523,  0.4525,  0.0062]]], grad_fn=<ViewBackward0>)


In [44]:
top_k_logits, top_k_indices = logits.topk(top_k, dim = -1) #get top k experts
top_k_logits, top_k_indices

(tensor([[[ 0.0238, -0.2771],
          [ 0.1839, -0.5727],
          [ 1.5661,  0.8137],
          [ 0.6523,  0.4525]]], grad_fn=<TopkBackward0>),
 tensor([[[0, 1],
          [2, 0],
          [2, 0],
          [0, 1]]]))

use -inf and apply softmax.

In [45]:
# full_like clones a tensor and fills it with a specifiedvalue.
zeros = torch.full_like(logits, float('-inf'))
sparse_logits = zeros.scatter(-1, top_k_indices, top_k_logits)
sparse_logits

tensor([[[ 0.0238, -0.2771,    -inf],
         [-0.5727,    -inf,  0.1839],
         [ 0.8137,    -inf,  1.5661],
         [ 0.6523,  0.4525,    -inf]]], grad_fn=<ScatterBackward0>)

In [46]:
gating_output = F.softmax(sparse_logits, dim = -1)
gating_output

tensor([[[0.5747, 0.4253, 0.0000],
         [0.3194, 0.0000, 0.6806],
         [0.3203, 0.0000, 0.6797],
         [0.5498, 0.4502, 0.0000]]], grad_fn=<SoftmaxBackward0>)

Create a class for TopKRouting

In [47]:
class TopKRouter(nn.Module):
  def __init__(self, n_embd, num_experts, top_k):
    super(TopKRouter, self).__init__()
    self.top_k = top_k
    self.Linear = nn.Linear(n_embd, num_experts)

  def forward(self, mh_output):
    # mh_output is the output tensor from multihead self attention block
    logits = self.Linear(mh_output)
    top_k_logits, indices = logits.topk(self.top_k, dim = -1)
    zeros = torch.full_like(logits, float('-inf'))
    sparse_logits = zeros.scatter(-1, indices, top_k_logits)
    router_output = F.softmax(sparse_logits, dim = -1)
    return router_output, indices

Testing it

In [48]:
num_experts = 3
top_k = 2
n_embd = 8

mh_output = torch.randn(1,4,n_embd)
top_k_gate = TopKRouter(n_embd, num_experts, top_k)
gating_output, indices = top_k_gate(mh_output)
gating_output.shape, gating_output, indices

(torch.Size([1, 4, 3]),
 tensor([[[0.6177, 0.0000, 0.3823],
          [0.6445, 0.3555, 0.0000],
          [0.0000, 0.3600, 0.6400],
          [0.5666, 0.4334, 0.0000]]], grad_fn=<SoftmaxBackward0>),
 tensor([[[0, 2],
          [0, 1],
          [2, 1],
          [0, 1]]]))

Noisy top-k gating is an important tool in training MoE models.
Essentially, u dont want all the tokens to be sent to the same set of 'favoured' experts.
You want a fine balance of exploitation and exploration. For this purpose, to load balance, it is helpful to add standard noise to logits from the gating linear layer. This makes training more efficient.

Creating NoisyTopKRouter class

In [49]:
class NoisyTopKRouter(nn.Module):
  def __init__(self, n_embd, num_experts, top_k):
    super(NoisyTopKRouter, self).__init__()
    self.top_k = top_k
    #layer for router logits
    self.topkroute_linear = nn.Linear(n_embd, num_experts)
    self.noise_linear = nn.Linear(n_embd, num_experts)

  def forward(self, mh_output):
    logits = self.topkroute_linear(mh_output)
    #Noise logits
    noise_logits = self.noise_linear(mh_output)
    #Adding scaled unit gaussian noise to logits
    noise = torch.randn_like(logits)*F.softplus(noise_logits)
    noisy_logits = logits + noise

    top_k_logits, indices = noisy_logits.topk(self.top_k, dim = -1)
    zeros = torch.full_like(noisy_logits, float('-inf'))
    sparse_logits = zeros.scatter(-1, indices, top_k_logits)
    router_output = F.softmax(sparse_logits, dim = -1)
    return router_output, indices

In [50]:
num_experts = 3
top_k = 2
n_embd = 8

mh_output = torch.randn(1,4,n_embd)
noisy_top_k_gate = NoisyTopKRouter(n_embd, num_experts, top_k)
gating_output, indices = noisy_top_k_gate(mh_output)
gating_output.shape, gating_output, indices

(torch.Size([1, 4, 3]),
 tensor([[[0.6890, 0.0000, 0.3110],
          [0.0000, 0.2910, 0.7090],
          [0.0000, 0.3835, 0.6165],
          [0.0000, 0.4379, 0.5621]]], grad_fn=<SoftmaxBackward0>),
 tensor([[[0, 2],
          [2, 1],
          [2, 1],
          [2, 1]]]))

Creating a sparse MoE module

In [51]:
class SparseMoE(nn.Module):
  def __init__(self, n_embd, num_experts, top_k):
    super(SparseMoE, self).__init__()
    self.router = NoisyTopKRouter(n_embd, num_experts, top_k)
    self.experts = nn.ModuleList([Expert(n_embd) for _ in range(num_experts)])
    self.top_k = top_k

  def forward(self, x):
    batch_size, sequence_length, _ = x.shape # Explicitly get batch_size and sequence_length from input x
    grating_output, indices = self.router(x)
    final_output = torch.zeros_like(x)

    #Reshape inputs for batch processing using explicit dimensions
    flat_x = x.view(batch_size * sequence_length, -1)
    # Force flat_gating_output to have consistent first dimension with flat_x
    flat_gating_output = grating_output.view(batch_size * sequence_length, grating_output.size(-1))

    #Process each expert in parallel
    for i, expert in enumerate(self.experts):
      # Create a mask for the inputs where the current expert is in top_k
      expert_mask = (indices == i).any(dim=-1) # (batch_size, sequence_length)
      flat_mask = expert_mask.view(batch_size * sequence_length) # (batch_size * sequence_length,)

      if flat_mask.any():
        expert_input = flat_x[flat_mask]
        expert_output = expert(expert_input)

        #Extract and apply gating scores
        gating_scores = flat_gating_output[flat_mask, i].unsqueeze(1)
        weighted_output = expert_output * gating_scores

        #Update final output additively by indexing and adding
        temp_output = torch.zeros_like(final_output)
        # Flatten temp_output and update elements based on flat_mask
        temp_output.view(batch_size * sequence_length, -1)[flat_mask] = weighted_output
        final_output += temp_output

    return final_output, indices

In [52]:
num_experts = 3
top_k = 2
n_embd = 8
dropout = 0.1

mh_output = torch.randn(1,8,n_embd)
sparse_moe = SparseMoE(n_embd, num_experts, top_k)
final_output, indices = sparse_moe(mh_output)
print("Shape of the final output:", final_output.shape)
print(final_output)

Shape of the final output: torch.Size([1, 8, 8])
tensor([[[ 0.1115, -0.4054,  0.0801,  0.1994,  0.1420,  0.0347, -0.6669,
           0.1496],
         [ 0.1613, -0.0793, -0.2446,  0.4436,  0.0635,  0.0302, -0.5552,
           0.0203],
         [ 0.1008,  0.0977, -0.0308, -0.0594, -0.0378, -0.0561, -0.5368,
           0.1341],
         [ 0.0867, -0.4320, -0.0776,  0.1290, -0.1099,  0.0828, -0.3973,
           0.1056],
         [ 0.0000, -0.0628,  0.0606,  0.1844, -0.1546,  0.0913, -0.8794,
           0.4309],
         [ 0.1018,  0.0410,  0.0444, -0.0417,  0.1514,  0.1215, -0.2027,
          -0.2926],
         [ 0.0921,  0.0734, -0.0875, -0.0399, -0.0321, -0.1383, -0.2201,
           0.2548],
         [ 0.1873, -0.2648, -0.1611,  0.1479, -0.0594, -0.0845, -0.1027,
          -0.0178]]], grad_fn=<AddBackward0>)
